In [12]:
from pathlib import Path

import os
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.dataset as ds

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

In [3]:
BASE = Path(r"G:\My Drive\HFT\Capital Stake Day\parsed\2026-06-30")

F_SNAPSHOT = BASE / "2026-06-30_ob_snapshot.parquet"
F_UPDATES  = BASE / "2026-06-30_ob_updates.parquet"
F_TRADES   = BASE / "2026-06-30_trades.parquet"

SYMBOL = "MCB"

In [5]:
def read_symbol_chunked(path: Path, symbol: str, batch_size: int = 500_000) -> pd.DataFrame:
    """Stream the file batch-by-batch, filter each batch, concat survivors.
    Peak memory ≈ one batch + accumulated MCB rows."""
    pf = pq.ParquetFile(path)
    kept = []
    for batch in pf.iter_batches(batch_size=batch_size):
        mask = pa.compute.equal(batch.column("symbol"), symbol)
        filtered = batch.filter(mask)
        if filtered.num_rows:
            kept.append(filtered)
    if not kept:
        # empty frame with correct schema
        return pf.schema_arrow.empty_table().to_pandas()
    return pa.Table.from_batches(kept).to_pandas()

df_trades   = read_symbol(F_TRADES,   SYMBOL)
df_updates  = read_symbol(F_UPDATES,  SYMBOL)
df_snapshot = read_symbol(F_SNAPSHOT, SYMBOL)

print(f"trades:   {df_trades.shape}")
print(f"updates:  {df_updates.shape}")
print(f"snapshot: {df_snapshot.shape}")

trades:   (864, 20)
updates:  (2294, 22)
snapshot: (64721, 27)


In [17]:
df_trades.to_csv(os.path.join(BASE, f"trades_{SYMBOL}.csv"))
df_updates.to_csv(os.path.join(BASE, f"Ob_updates_{SYMBOL}.csv"))
df_snapshot.to_csv(os.path.join(BASE, f"Ob_snapshot_{SYMBOL}.csv"))



In [18]:
def add_time_cols(df: pd.DataFrame, primary: str = "capture_ts") -> pd.DataFrame:
    """ts_ms  : int64 milliseconds since epoch — canonical comparison/join key.
    t_day_s: float seconds since midnight UTC — human inspection only.
    .dt.as_unit('ns') guards against pandas 3.0 parsing to datetime64[us],
    where .astype('int64') would return MICROseconds and silently break scale."""
    ts = pd.to_datetime(df[primary], utc=True, format="ISO8601").dt.as_unit("ns")
    df["ts_ms"] = ts.astype("int64") // 1_000_000
    df["t_day_s"] = (ts - ts.dt.normalize()).dt.total_seconds()
    return df

df_updates  = add_time_cols(df_updates)    # capture_ts — ms
df_snapshot = add_time_cols(df_snapshot)   # capture_ts — ms (do NOT use orig_time: 1s precision)
df_trades   = add_time_cols(df_trades)

In [ ]:
import ast                          # parses the stringified tuple in trades.resting_order_id, e.g. "('0010THF...', 407.31)"
from dataclasses import dataclass   # lightweight struct for one resting order

# =========================================================================
# BLOCK 1 — TIME KEYS
# Two clocks, two jobs:
#   ts_exch = exchange/matching-engine time  -> used ONLY to sequence events
#   ts_cap  = your wall-clock receive time   -> used ONLY to decide when your
#             strategy could have acted (no-lookahead clock)
# =========================================================================
def to_ms(df, col):
    # Parse the ISO strings to timezone-aware datetimes.
    # .dt.as_unit("ns") is the pandas-3.0 guard: pandas 3 parses these as
    # datetime64[us], and .astype("int64") on that would return MICROseconds.
    # Forcing ns first means the integer is always nanoseconds...
    ts = pd.to_datetime(df[col], utc=True).dt.as_unit("ns")
    # ...which we then integer-divide down to milliseconds. Your feed is
    # ms-precision throughout (measured), so int64 milliseconds is exact.
    return ts.astype("int64") // 1_000_000

# Exchange-side event clocks. Updates/trades: tag 60 (TransactTime).
df_updates["ts_exch"]  = to_ms(df_updates, "transact_time")
df_trades["ts_exch"]   = to_ms(df_trades, "transact_time")
# Snapshots have no tag 60; orig_time (tag 42) is their exchange clock.
# It's truncated to whole seconds in this feed — acceptable, because a
# snapshot is a full-state reconcile: if it lands up to 1s early/late, the
# very next events overwrite/correct any transient error.
df_snapshot["ts_exch"] = to_ms(df_snapshot, "orig_time")

# capture_ts exists in all three tables at ms precision. Attach it everywhere
# so every replayed book state can carry "earliest moment I knew this".
for df in (df_updates, df_trades, df_snapshot):
    df["ts_cap"] = to_ms(df, "capture_ts")

# =========================================================================
# BLOCK 2 — RESOLVE THE RESTING ORDER ON EACH TRADE
# trades.resting_order_id looks like "('0010THF0D0000R0K', 407.31)" when the
# parser could resolve it (282/864 rows in your sample), else NaN.
# ast.literal_eval safely turns that string into a real Python tuple;
# [0] takes the order id and drops the price. Non-strings (NaN) -> None.
# =========================================================================
df_trades["rest_oid"] = df_trades["resting_order_id"].map(
    lambda x: ast.literal_eval(x)[0] if isinstance(x, str) and x.startswith("(") else None)

@dataclass
class Order:
    side: str      # 'BUY' or 'SELL'
    price: float
    qty: float     # remaining (unfilled) quantity

# =========================================================================
# BLOCK 3 — THE ORDER BOOK
# State = one dict: order_id -> Order. Everything (adds, cancels, fills,
# snapshots) is expressed as mutations of this dict; price levels are
# derived on demand in depth(). Order-level (not level-level) state is what
# lets a snapshot reconcile precisely.
# =========================================================================
class Book:
    def __init__(self):
        self.o: dict[str, Order] = {}          # the entire book lives here

    # ---- ob_updates: event == "ORDER_ADD" ----
    def add(self, r):
        if pd.notna(r.order_id):               # defensive: skip malformed adds
            # Insert or overwrite. Overwrite matters: after a snapshot, an
            # order may already exist under this id; the add's values win.
            self.o[str(r.order_id)] = Order(r.side, float(r.price), float(r.qty))

    # ---- ob_updates: event == "CANCEL" ----
    def cancel(self, r):
        # pop with default=None: a cancel for an id we never saw (order was
        # resting before your capture window began) is silently tolerated —
        # 150 of these in your sample, all pre-open leftovers.
        self.o.pop(str(r.order_id), None)

    # ---- trades: a fill REMOVES resting liquidity from the book ----
    def trade(self, r):
        oid = r.rest_oid
        if oid and oid in self.o:
            # Exact path: we know which resting order got hit. Decrement it;
            # delete when fully consumed (<= 0 also covers overfills caused
            # by a stale snapshot qty).
            self.o[oid].qty -= float(r.qty)
            if self.o[oid].qty <= 0:
                del self.o[oid]
            return
        # Fallback path (the ~2/3 of trades with unresolvable id):
        # the passive side is the opposite of the aggressor.
        passive = {"BUY": "SELL", "SELL": "BUY"}.get(r.aggressor_side)
        if passive is None:
            return  # AUCTION print: no unique passive side; the next 5s
                    # snapshot reconciles whatever the cross consumed.
        # Consume r.qty from orders resting at exactly the trade price on the
        # passive side. Iteration order over the dict is arbitrary — we don't
        # know true queue priority for these — but for best-bid/ask purposes
        # only the LEVEL total matters, and that is decremented correctly.
        # list(...) snapshot lets us delete keys while iterating.
        rem = float(r.qty)
        for k, o in list(self.o.items()):
            if o.side == passive and o.price == float(r.price):
                take = min(o.qty, rem)         # can't take more than the order has
                o.qty -= take
                rem  -= take
                if o.qty <= 0:
                    del self.o[k]
                if rem <= 0:
                    break                      # trade fully accounted for

    # ---- ob_snapshot: authoritative full state, replaces the book ----
    def snapshot(self, rows):
        # `rows` = all BID/OFFER rows sharing one msg_seq (one 35=W message).
        tgt = {}                               # build the target book from scratch
        for r in rows.itertuples():
            side = "BUY" if r.entry_type == "BID" else "SELL"
            px = float(r.px)
            disclosed = 0.0                    # sum of individually listed orders
            if isinstance(r.order_ids, str) and r.order_ids:
                # order_ids / order_qtys are pipe-separated parallel lists:
                # "ORD001|ORD002" / "500|1200" -> two orders at this level.
                for oid, q in zip(r.order_ids.split("|"), str(r.order_qtys).split("|")):
                    tgt[oid] = Order(side, px, float(q))
                    disclosed += float(q)
            # Level qty can exceed the disclosed sum (undisclosed orders:
            # n_orders_at_level > n_orders_detailed). Park the residual in a
            # synthetic order so depth() still reports the true level total.
            # Key is deterministic per (side, price) so successive snapshots
            # overwrite rather than accumulate.
            if float(r.qty) - disclosed > 0:
                tgt[f"__H_{side}_{px}"] = Order(side, px, float(r.qty) - disclosed)
        # Wholesale replace = "add what's missing, remove what's stale,
        # correct every qty" in one assignment. This is the reconcile step
        # that erases any drift accumulated since the last snapshot.
        self.o = tgt

    # ---- derived view: aggregate orders into top-n price levels ----
    def depth(self, n=5):
        bids, asks = {}, {}                    # price -> total qty
        for o in self.o.values():
            d = bids if o.side == "BUY" else asks
            d[o.price] = d.get(o.price, 0.0) + o.qty
        out = {}
        # Bids: best = highest price, so sort descending. Asks: ascending.
        for i, p in enumerate(sorted(bids, reverse=True)[:n], 1):
            out[f"bid_px_{i}"], out[f"bid_qty_{i}"] = p, bids[p]
        for i, p in enumerate(sorted(asks)[:n], 1):
            out[f"ask_px_{i}"], out[f"ask_qty_{i}"] = p, asks[p]
        return out                             # flat dict -> one row of book_states

# =========================================================================
# BLOCK 4 — SNAPSHOT MESSAGES AS SINGLE EVENTS
# The snapshot table is long-format: ~20-40 rows per 35=W message (one per
# side/level/entry-type). The book only wants BID/OFFER rows, grouped so one
# msg_seq = one event.
# =========================================================================
snap_book = df_snapshot[df_snapshot["entry_type"].isin(["BID", "OFFER"])]
# Pre-split into {msg_seq: DataFrame-of-its-rows} once, so the replay loop
# does an O(1) dict lookup instead of filtering 64k rows per snapshot.
snap_groups = dict(tuple(snap_book.groupby("msg_seq")))
# One (ts_exch, ts_cap) pair per snapshot message for the event stream.
snap_ev = snap_book.groupby("msg_seq", as_index=False)[["ts_exch", "ts_cap"]].min()

# =========================================================================
# BLOCK 5 — UNIFIED EVENT STREAM
# Each event is a sortable tuple: (ts_exch, kind_rank, appl_seq, kind, payload)
#   ts_exch   — primary key: exchange time (the verified correct clock)
#   kind_rank — tie-break 1: snapshots get 0, incrementals get 1, so at equal
#               time the snapshot applies FIRST (it describes state as-of
#               that instant; subsequent events build on it)
#   appl_seq  — tie-break 2: updates and trades share channel 2011 and their
#               combined appl_seq is strictly increasing (verified), so this
#               reproduces exact wire order within a same-ms burst. Snapshots
#               use 0 (they're on channel 1011; appl_seq isn't comparable).
# =========================================================================
events  = [(r.ts_exch, 1, r.appl_seq, "U", r) for r in df_updates.itertuples()]
events += [(r.ts_exch, 1, r.appl_seq, "T", r) for r in df_trades.itertuples()]
events += [(r.ts_exch, 0, 0,          "S", r) for r in snap_ev.itertuples()]
events.sort(key=lambda e: (e[0], e[1], e[2]))   # lexicographic: time, rank, seq

# =========================================================================
# BLOCK 6 — REPLAY
# Walk the stream once, mutate the book, record top-5 depth after EVERY event.
# =========================================================================
book, states = Book(), []
for ts_exch, _, _, kind, obj in events:          # _ = rank and seq, done their job in the sort
    if kind == "S":
        book.snapshot(snap_groups[obj.msg_seq])  # obj here is a snap_ev row -> look up its level rows
        ts_cap = obj.ts_cap
    elif kind == "U":
        # One-liner dispatch: pick the bound method by event type, call with row.
        (book.add if obj.event == "ORDER_ADD" else book.cancel)(obj)
        ts_cap = obj.ts_cap
    else:                                        # kind == "T"
        book.trade(obj)
        ts_cap = obj.ts_cap
    # Record state AFTER applying the event. ts_exch = when it happened at
    # the exchange; ts_cap = when you received it (earliest actionable time).
    states.append({"ts_exch": ts_exch, "ts_cap": ts_cap, "kind": kind, **book.depth()})

book_states = pd.DataFrame(states)
# NOTE for the strategy layer: join your decisions on ts_cap (no-lookahead).
# ts_cap is NOT guaranteed monotonic across rows — that's receipt jitter
# itself — so before merge_asof either sort_values("ts_cap") or use
# book_states["ts_cap"].cummax() as the join key.

In [19]:
df_updates.tail()

,transact_time,sending_time,capture_ts,msg_seq,appl_seq,channel,segment,market,event,exec_type_code,exec_type,exec_inst,symbol,side_code,side,price,qty,order_id,buy_ref,sell_ref,resting_ref,raw_last_px,ts_ms,t_day_s
2289,2026-06-30 10:29:33.990000+00:00,2026-06-30 10:29:34.007000+00:00,2026-06-30 10:29:34.079000+00:00,293421,1165347,2011,011,REG,CANCEL,4,CANCELLED,<NA>,MCB,2,SELL,405.95,8498.0,0010THF0D0017GEN,0,1145275,1145275,0.0,1782815374079,37774.079
2290,2026-06-30 10:29:34.500000+00:00,2026-06-30 10:29:34.517000+00:00,2026-06-30 10:29:34.588000+00:00,293542,1165408,2011,011,REG,CANCEL,4,CANCELLED,<NA>,MCB,2,SELL,405.69,2837.0,0010THF0D0017TYU,0,1162764,1162764,0.0,1782815374588,37774.588
2291,2026-06-30 10:29:44.840000+00:00,2026-06-30 10:29:44.854000+00:00,2026-06-30 10:29:44.924000+00:00,296660,1167252,2011,011,REG,CANCEL,4,CANCELLED,<NA>,MCB,2,SELL,405.94,2000.0,0010THF0D0017OPO,0,1155781,1155781,0.0,1782815384924,37784.924
2292,2026-06-30 10:29:52.020000+00:00,2026-06-30 10:29:52.036000+00:00,2026-06-30 10:29:52.106000+00:00,298597,1168346,2011,011,REG,ORDER_ADD,<NA>,<NA>,<NA>,MCB,1,BUY,405.10,100.0,0010THF0D0017XRC,<NA>,<NA>,<NA>,NaN,1782815392106,37792.106
2293,2026-06-30 10:29:52.030000+00:00,2026-06-30 10:29:52.038000+00:00,2026-06-30 10:29:52.109000+00:00,298599,1168348,2011,011,REG,ORDER_ADD,<NA>,<NA>,<NA>,MCB,2,SELL,405.19,2837.0,0010THF0D0017XRE,<NA>,<NA>,<NA>,NaN,1782815392109,37792.109


In [21]:
df_snapshot.tail()

,snapshot_time,orig_time,capture_ts,msg_seq,channel,segment,market,symbol,trading_status,phase,suspended_all_day,break_reason,prev_close,num_trades,cum_volume,cum_value,entry_type_code,entry_type,level,px,qty,n_orders_at_level,n_orders_detailed,order_ids,order_qtys,visible_qty_sum,is_xf_tick_floor,ts_ms,t_day_s
64716,2026-06-30 11:52:03.489000+00:00,2026-06-30 11:52:03+00:00,2026-06-30 11:52:03.560000+00:00,2508,1011,010,REG,MCB,E0,MARKET_CLOSED,False,<NA>,405.01,878,374981.0,1.524348e+08,x1,NET_CHANGE_1,0,0.50,0.0,0,<NA>,<NA>,<NA>,NaN,None,1782820323560,42723.56
64717,2026-06-30 11:52:03.489000+00:00,2026-06-30 11:52:03+00:00,2026-06-30 11:52:03.560000+00:00,2508,1011,010,REG,MCB,E0,MARKET_CLOSED,False,<NA>,405.01,878,374981.0,1.524348e+08,x2,NET_CHANGE_2,0,0.51,0.0,0,<NA>,<NA>,<NA>,NaN,None,1782820323560,42723.56
64718,2026-06-30 11:52:03.489000+00:00,2026-06-30 11:52:03+00:00,2026-06-30 11:52:03.560000+00:00,2508,1011,010,REG,MCB,E0,MARKET_CLOSED,False,<NA>,405.01,878,374981.0,1.524348e+08,xe,UPPER_CIRCUIT_BREAKER,0,445.51,0.0,0,<NA>,<NA>,<NA>,NaN,None,1782820323560,42723.56
64719,2026-06-30 11:52:03.489000+00:00,2026-06-30 11:52:03+00:00,2026-06-30 11:52:03.560000+00:00,2508,1011,010,REG,MCB,E0,MARKET_CLOSED,False,<NA>,405.01,878,374981.0,1.524348e+08,xf,LOWER_CIRCUIT_BREAKER,0,364.51,0.0,0,<NA>,<NA>,<NA>,NaN,False,1782820323560,42723.56
64720,2026-06-30 11:52:03.489000+00:00,2026-06-30 11:52:03+00:00,2026-06-30 11:52:03.560000+00:00,2508,1011,010,REG,MCB,E0,MARKET_CLOSED,False,<NA>,405.01,878,374981.0,1.524348e+08,5,CLOSING_PRICE,0,405.51,0.0,0,<NA>,<NA>,<NA>,NaN,None,1782820323560,42723.56


In [20]:
df_trades.tail()

,transact_time,sending_time,capture_ts,msg_seq,appl_seq,channel,segment,market,exec_type_code,exec_type,exec_inst,symbol,price,qty,buy_ref,sell_ref,resting_ref,resting_order_id,initiator,aggressor_side,ts_ms,t_day_s
859,2026-06-30 10:29:51.870000+00:00,2026-06-30 10:29:51.887000+00:00,2026-06-30 10:29:51.957000+00:00,298564,1168321,2011,011,REG,F,TRADE,<NA>,MCB,405.02,2000.0,1111635,0,1111635,"('0010THF0D0016LEW', 405.02)",SELLER_INITIATED,SELL,1782815391957,37791.957
860,2026-06-30 10:29:51.870000+00:00,2026-06-30 10:29:51.887000+00:00,2026-06-30 10:29:51.957000+00:00,298565,1168322,2011,011,REG,F,TRADE,<NA>,MCB,405.01,20.0,665732,0,665732,"('0010THF0D000RA9O', 405.01)",SELLER_INITIATED,SELL,1782815391957,37791.957
861,2026-06-30 10:29:51.870000+00:00,2026-06-30 10:29:51.887000+00:00,2026-06-30 10:29:51.957000+00:00,298566,1168323,2011,011,REG,F,TRADE,<NA>,MCB,405.00,3936.0,443507,0,443507,"('0010THF0D000HC6J', 405.0)",SELLER_INITIATED,SELL,1782815391957,37791.957
862,2026-06-30 10:29:54.040000+00:00,2026-06-30 10:29:54.054000+00:00,2026-06-30 10:29:54.124000+00:00,299146,1168662,2011,011,REG,F,TRADE,<NA>,MCB,405.10,100.0,1168346,0,1168346,<NA>,SELLER_INITIATED,SELL,1782815394124,37794.124
863,2026-06-30 10:29:54.040000+00:00,2026-06-30 10:29:54.054000+00:00,2026-06-30 10:29:54.125000+00:00,299147,1168663,2011,011,REG,F,TRADE,<NA>,MCB,405.00,7504.0,443507,0,443507,"('0010THF0D000HC6J', 405.0)",SELLER_INITIATED,SELL,1782815394125,37794.125
